<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [16]</a>'.</span>

# 0.0. Importing the Modules 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools
import multiprocessing
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from xgboost import XGBRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, ConstantKernel as C



# 1. Data Processing for the Model Development

## 1.1. Introducing the Collected Data

In [ ]:
Data = pd.read_excel('Train_Test_Data_with_Descriptor.xlsx') # Reading the data from excel file
Columns_to_Drop = ['Metal', 'Surface', 'Reagent A', 'Reagent B', 'Product C', 'Product D', 'Code'] # Looking for the columns not require for model training
Input_Values = Data.drop(Columns_to_Drop, axis = 1) # Droping the unnecessary columns. Stored the values as DataFrame in matrix called Input_Values
# Input_Values.head()

## 1.2. Introducing the 'None' Values Where No vdW Correction is Used.

In [ ]:
Input_Values['vdW Correction'] = Input_Values['vdW Correction'].apply(lambda x: 'None' if pd.isna(x) else x) # Looking fot the entries entries in vdW Correction column and replacing empty values with 'none'
# Input_Values.head()

## 1.3. Droping the Columns with All the Entrties Havin Zero Value to Enhance the Performance

In [ ]:
# Droping the columns with all the zero entrties to enhance the performance
Zero_Columns = Input_Values.columns[(Input_Values == 0).all()] # Looking for the columns having zero values. 
print(Zero_Columns) # Printing the columns with zero values 
Input_Values = Input_Values.drop(Zero_Columns, axis = 1) # Droping the columns with zero values

## 1.4. Shifting the Activation Barrier to the First Column

In [ ]:
column_to_shift = 'Activation Barrier_eV' # Listing the head of column I want to shift
Activation_Barrier = Input_Values.pop(column_to_shift) # Removing the column containing the activation energy from the dataset 
Input_Values.insert(0, column_to_shift, Activation_Barrier) # Inserting the removed column at initial position without changing the column header
# Input_Values.head() # Looking for the entries in Input_Values DataFrame

## 1.5. Looking for the Categorical Columns

In [ ]:
Input_Values.dtypes # Checking the data type of all the columns in DataFrame Input_Values
Categorical_Columns = Input_Values.select_dtypes(include='object').columns # Selecting the columns with datatype 'Object'. This columns are the classified values
# print(Categorical_Columns) # Printing the columns having Categorical_Columns

## 1.6. Applying One Hot Encoding to the Classified Values to Convert into Numerical Form

In [ ]:
from sklearn.preprocessing import OneHotEncoder # Inroducing the one-hot encoder
OHE = OneHotEncoder(sparse_output=False) # Defining one-hot encoder as OHE
Encoded_Values = OHE.fit_transform(Input_Values[Categorical_Columns]) # Encoding the categorical columns to numerical values
Encoded_Values_df = pd.DataFrame(Encoded_Values, columns=OHE.get_feature_names_out(Categorical_Columns)) # Converting the encoded values array to DataFrame
Encoded_Values.shape # Checking the shape of the Encoded_Values
print(type(Encoded_Values_df)) # Checking the type of the data frame

## 1.7. Replacing the Classified Data in Original Data with Encoded Data

In [ ]:
Input_Values_Classifier_Values_Drop = Input_Values.drop(columns= Categorical_Columns) # Droping the columns having categorical values from the original data
Input_Values_Classifier_Values_Drop.shape # Checking the shape of the DataFrame after dropping the categorical columns

Input_Values_with_Encoded_Data = pd.concat((Input_Values_Classifier_Values_Drop, Encoded_Values_df), axis = 1) # Concatinating the original data after droping categorical columns with encoded values
Input_Values_with_Encoded_Data.shape # Checking the shape of the concatinated new data frame
print(type(Input_Values_with_Encoded_Data)) # Printing the type of concatenated data frame

## 1.8. Checking for the Empty Cells in the Modified Dataset

In [ ]:
NaN_in_Data = Input_Values_with_Encoded_Data.isna().any().any() # Looking for the Not a Number in a DataFrame
print(NaN_in_Data) # Printing the boolian values

## 1.9. Splitting Datasent into the Train and Test Set for the Model Development

In [ ]:
from sklearn.model_selection import train_test_split # Importing the train test split funciton to split the dataset
X_Train, X_Test, Y_Train, Y_Test = train_test_split(Input_Values_with_Encoded_Data.iloc[:, 1:], Input_Values_with_Encoded_Data.iloc[:,0], test_size=0.10, random_state=42) # Spliting the data in 7:3 ratio of train to test split

## 1.10. Checking the Indices of Data Splitted Into Training and Test Dataset

In [ ]:
X_Training_Indices = X_Train.index.tolist() # Looking for the indices of training data and storing them into list
X_Test_Indices = X_Test.index.tolist() # Looking for the indices of testing data and storing them into list
print((X_Training_Indices)) # Printing the total entries of the training dataset
print((X_Test_Indices)) # Printing the total entries of the testing dataset

## 1.11. Storing the Training and Testing Data in Terms of Reaction Involved

In [ ]:
Training_Data = Data.iloc[X_Training_Indices].copy() # Looking for the training data based on indices in original dataset
Testing_Data = Data.iloc[X_Test_Indices].copy() # looking for the testing data based on indices in original dataset

Training_Data['Reaction'] = (Training_Data.iloc[:,7].astype(str) + ' + ' + Training_Data.iloc[:,8].astype(str) + '  ===>  ' + Training_Data.iloc[:,9].astype(str) + ' + ' + Training_Data.iloc[:,10].astype(str)) # Writing the reaction for training data introducing new column called 'reaction'
Testing_Data['Reaction'] = (Testing_Data.iloc[:,7].astype(str) + ' + ' + Testing_Data.iloc[:,8].astype(str) + '  ===>  ' + Testing_Data.iloc[:,9].astype(str) + ' + ' + Testing_Data.iloc[:,10].astype(str)) # Writing the reaction for testing data introducing new column called 'reaction'

columns_to_keep = [0, 1, 2, 5, 7, 8, 9, 10, -1] # Introducing a list of columns to keep in training and testing data from the original data

Training_Data = Training_Data.iloc[:, columns_to_keep] # Regenetating the training data keeping necessary columns only for output analysis pupose
Testing_Data = Testing_Data.iloc[:, columns_to_keep] # Regenetating the testing data keeping necessary columns only for output analysis purpose

Data_with_Reactions  = pd.concat([Training_Data, Testing_Data], axis=0) # Creating the new DataFrame combining training and testing data called Data_with_Reaction for analysis purpose
Data_with_Reactions.to_excel('Data_With_Reaction.xlsx', index=False) # Converting the new DataFrame to Excel File

# 2.0. Hyperparameter Optimization

## 2.1. Describing the Algorithms for ML Model

In [ ]:

models = {
    "LR": LinearRegression(),
    "RFR": RandomForestRegressor(random_state=42),
    "GBR": GradientBoostingRegressor(random_state=42),
    "XGBR": XGBRegressor(random_state=42),
    "DTR": DecisionTreeRegressor(random_state=42),
    "ETR": ExtraTreesRegressor(random_state=42),
    "SVR_M": SVR(),
    "KRR": KernelRidge(),
    "KNNR": KNeighborsRegressor(),
    "GPR": GaussianProcessRegressor()
}

## 2.2.A Creating a Kernel Options for the GPR

In [ ]:
kernel_options = [
    C(1.0, (1e-3, 1e3)) * RBF(length_scale=1.0, length_scale_bounds=(1e-8, 1e8)),
    C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, length_scale_bounds=(1e-8, 1e8), nu=1.5),
    C(1.0, (1e-3, 1e3)) * RationalQuadratic(length_scale=1.0, alpha=1.0)
]

# 2.2. Creating the Hyperparameter Grid for the Models

In [ ]:
param_grid = {
    # 🔹 Linear Regression (No hyperparameters to tune)
    "LR": {},  

    # 🔹 Random Forest Regressor
    "RFR": {
        "n_estimators": [500, 1000, 1500, 2000, 2500, 3000],  
        "max_depth": [5, 10, 15, 20, 25, 30],  
    },

    # 🔹 Gradient Boosting Regressor
    "GBR": {
        "n_estimators": [500, 1000, 1500, 2000, 2500, 3000],  
        "learning_rate": [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1],  
        "max_depth": [5, 10, 15, 20, 25, 30],   
    },

    # 🔹 XGBoost Regressor
    "XGBR": {
        "n_estimators": [500, 1000, 1500, 2000, 2500, 3000],  
        "learning_rate": [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1],  
        "max_depth": [5, 10, 15, 20, 25, 30],  
        'min_child_weight': [1, 3, 5, 7, 9]
    },

    # 🔹 Decision Tree Regressor
    "DTR": {
        "max_depth": [5, 10, 15, 20, 25, 30]
    },

    # 🔹 Extra Trees Regressor
    "ETR": {
        "n_estimators": [500, 1000, 1500, 2000, 2500, 3000],  
        "max_depth": [5, 10, 15, 20, 25, 30],    
    },

    # 🔹 Support Vector Regressor
    "SVR_M": {
        "kernel": ["linear", "poly", "rbf", "sigmoid"],  
        "C": [0.1, 1, 10, 100],  
        "gamma": ["scale", "auto"],  
        "degree": [2, 3, 4]
    },

    # 🔹 Kernel Ridge Regression
    "KRR": {  
        "kernel": ["linear", "poly", "rbf", "sigmoid"],  
        "alpha": [0.01, 0.1, 1, 10, 100],
        "degree": [2, 3, 4]
    },

    # 🔹 K-Nearest Neighbors Regressor
    "KNNR": {
        "n_neighbors": range(1, 30),  
        "weights": ["uniform", "distance"],  
        "algorithm": ["auto", "ball_tree", "kd_tree", "brute"],  
        "p": [1, 2]  
    },

    # 🔹 Gaussian Process Regressor
    "GPR": {
        "kernel": kernel_options,
        "alpha": [ 1e-3, 1e-2, 0.1, 0.2, 0.5, 1, 10, 100]
    },

    
}

## 2.3.Performing the Grid Search Cross Validation for Hyperparameter Optimization

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [ ]:
import pandas as pd

best_models = []
all_results = []  # List to store all models' parameter results

for model_name, model in models.items():
    print(f"🔄 Running Grid Search for {model_name}...")

    grid = GridSearchCV(model, param_grid[model_name], cv=5, scoring="neg_mean_absolute_error", n_jobs=-1, return_train_score=True)
    grid.fit(X_Train, Y_Train)

    # Store best model details
    best_models.append({
        "Model": model_name,
        "Best Score": grid.best_score_,
        "Best Params": grid.best_params_,
        "Best Estimator": grid.best_estimator_
    })

    # Store all parameter sets and scores
    for params, mean_test_score, mean_train_score in zip(grid.cv_results_["params"], grid.cv_results_["mean_test_score"], grid.cv_results_["mean_train_score"]):
        all_results.append({
            "Model": model_name,
            "Parameters": str(params),
            "Test Score": mean_test_score,
            "Train Score": mean_train_score
        })

# Convert to DataFrame
best_models_df = pd.DataFrame(best_models)
all_results_df = pd.DataFrame(all_results)

# Save to Excel (or CSV)
best_models_df.to_excel("Best_Models.xlsx", index=False)  # Save best models
all_results_df.to_excel("All_GridSearch_Results.xlsx", index=False)  # Save all grid search results

# Alternative: Save as CSV
best_models_df.to_csv("Best_Models.csv", index=False)
all_results_df.to_csv("All_GridSearch_Results.csv", index=False)

print("\n✅ Grid Search results saved to 'Best_Models.xlsx' and 'All_GridSearch_Results.xlsx'")

# Find the best model overall
best_model = max(best_models, key=lambda item: item["Best Score"])
best_estimator_among_all = best_model["Best Estimator"]
best_model_name = best_model["Model"]

print("\n🏆 **Best Overall Model:**")
print(f"🔹 Model: {best_model_name}")
print(f"🔹 Best Score: {best_model['Best Score']:.4f}")
print(f"🔹 Best Params: {best_model['Best Params']}")

# 10.0. Predicting the Reaction Energies for C2 Reactions on Ni and NiB Surface

## 10.1. Introducing the C2 Reaction Data for Activation Energy Prediction

In [ ]:
Data_to_Predict = pd.read_excel('Validation_Data_with_Descriptor.xlsx') #Reading the data from excel file
Columns_to_Drop = ['Metal', 'Surface', 'Reagent A', 'Reagent B', 'Product C', 'Product D', 'Code'] # Looking for the columns that does not require to train the model
In_Values = Data_to_Predict.drop(Columns_to_Drop, axis = 1) # Droping the unnecessary columns. Stored the values in DataFrame Input_Values

## 10.2. Introducing the 'None' Values Where No vdW Correction is Used.

In [ ]:
In_Values['vdW Correction'] = In_Values['vdW Correction'].apply(lambda x: 'None' if pd.isna(x) else x) #Looking fot the entries in vdW Correction column and replacing empty values with 'none'
In_Values.head()

## 10.3. Droping the Columns with All the Entrties Havin Zero Value to Enhance the Performance

In [ ]:
# # Droping the columns with all the zero entrties to enhance the performance
# Zero_Values = In_Values.columns[(In_Values == 0).all()] # Looking for the columns having zero values. 
# print(Zero_Values) # Printing the columns with zero values 
# In_Values = In_Values.drop(Zero_Values, axis = 1) # Droping the columns with zero values
In_Values = In_Values.drop(['delta_n_O', 'delta_n_H'], axis = 1)
# In_Values.head()


## 10.4. Shifting the Activation Barrier to the First Column

In [ ]:
column_to_shift = 'Activation Barrier_eV' # Assigning a string of column name which I want to shift
Activation_Barrier = In_Values.pop(column_to_shift) # Removing and returing the 
In_Values.insert(0, column_to_shift, Activation_Barrier) # Inserting the removed column Activation Barrier at initial position with header name column_to_shift which is nothing but 'Activation Barrier [eV]
In_Values.head() # Looking for the entries in Input_Values DataFrame

## 10.5. Looking for the Categorical Columns

In [ ]:
In_Values.dtypes # Checking the data type of all the columns in DataFrame Input_Values
Categorical_Data = In_Values.select_dtypes(include='object').columns # Selecting the columns with data type Object which are columns with classified values

## 10.6. Applying One Hot Encoding to the Classified Values to Convert into Numerical Form

In [ ]:
from sklearn.preprocessing import OneHotEncoder # Inroducing the one-hot encoder
OHE = OneHotEncoder(sparse_output=False) # Defining one-hot encoder as OHE
Encoded_Data = OHE.fit_transform(In_Values[Categorical_Data])#.toarray() # Encoding the Categorical columns to Encoded Values
# X_Test_Encoded_Values = OHE.fit_transform(X_test[['Functional', 'vdW Correction', 'Energy Term']]).toarray()
# X_Train_Encoded_Values.shape
Encoded_Data_df = pd.DataFrame(Encoded_Data, columns=OHE.get_feature_names_out(Categorical_Data)) # Converting the encoded values array to DataFrame
Encoded_Data.shape # Checking the shape of the Encoded_Values
print(type(Encoded_Values_df)) # Checking the type of the data frame

## 10.7. Introducing the Encoded Columns not Available in Prediction Dataset but Available in Training and Testing Dataset

In [ ]:
Columns_Encode_Train = Encoded_Values_df.columns # Writing the Columns of the Encoded Training Dataset


Columns_Encode_Prediction = Encoded_Data_df.columns # Writing the Columns of the Encoded Prediction Dataset


Uniuq_Columns_in_Training_Data = Columns_Encode_Train.difference(Columns_Encode_Prediction) # Comparing the Columns in Encoded Training and Testing Dataset


Rows_Required = Encoded_Data_df.shape[0] # Looking for Rows in Testing Data Set
Columns_Required = Uniuq_Columns_in_Training_Data.shape[0] # Looking for Columns needs to be Added in Encoded Test Data



Encoded_Data_to_Add = pd.DataFrame(index=range(Rows_Required), columns=range(Columns_Required)) # Generating the Dataframe to COncate with Encoded Test Set to Avoid Error of Missing Features While Testing
Encoded_Data_to_Add.fillna(value=0, inplace=True) # Adding the Zeros to All the Data as this values should be zero in encoding data
Encoded_Data_to_Add.columns = Uniuq_Columns_in_Training_Data # Introducing the Columns name for the Dataframe created


Encoded_Data_df = pd.concat([Encoded_Data_df, Encoded_Data_to_Add], axis=1) # Concatin the Encoded test set with zeros values of features not involved in Encoding process


Columns_Encode_Train_List = Columns_Encode_Train.to_list() # Convering the Index values of Columns to List
Encoded_Data_df = Encoded_Data_df.reindex(columns=Columns_Encode_Train_List) # Reindexing the columns of the generated Encoded Test Set in order to avoid an error of mismatching index with Encoded Training Data Set while testing


## 10.8. Replacing the Classified Data in Original Data with Encoded Data

In [ ]:
In_Values_Classifier_Values_Drop = In_Values.drop(columns= Categorical_Data) # Droping the columns having categorical values from the original data
In_Values_Classifier_Values_Drop.shape # Checking the shape of the DataFrame after dropping the categorical columns

In_Values_with_Encoded_Data = pd.concat((In_Values_Classifier_Values_Drop, Encoded_Data_df), axis = 1) # Concatinating the original values data from after droping categorical columns with encoded values
In_Values_with_Encoded_Data.shape # Checking the shape of the concatinated new data frame
print(type(In_Values_with_Encoded_Data)) # Printing the type of concatenated data frame

## 10.9. Splitting the X Values to Introduce for Activation Barrier Prediction

In [ ]:
X_Values_to_Model = In_Values_with_Encoded_Data.iloc[:,1:] # Defining the X_Values to introduce in the model

## 10.10. Predicting the Activation Barrier Using Different Algorithms

In [ ]:
DFT_Calculated_Values = In_Values_with_Encoded_Data.iloc[:, 0]  # Defining the DFT values

Best_Estimator_Y_Predicted_from_X_Data = []
Models = best_models_df["Model"]
estimators = best_models_df["Best Estimator"]
print(Models)
for estimator in estimators:
    predictions = estimator.predict(X_Values_to_Model)  # Predicting values
    Best_Estimator_Y_Predicted_from_X_Data.append(predictions)

# Convert list of predictions to DataFrame
Best_Estimator_Y_Predicted_from_X_Data_df = pd.DataFrame(Best_Estimator_Y_Predicted_from_X_Data).T  # Transpose to get predictions as columns

# Optionally name the columns
Best_Estimator_Y_Predicted_from_X_Data_df.columns = Models.values

# Concatenate with actual DFT values
Activation_Barrier_Comparred = pd.concat([DFT_Calculated_Values.reset_index(drop=True),Best_Estimator_Y_Predicted_from_X_Data_df],axis=1)

## 10.12. Storing the Predicted Activation Barrier in Excel Files

In [ ]:
# Reaction_Involved = Data_to_Predict.iloc[:,7:11]
Metals_Involved = Data_to_Predict.iloc[:,2]

Reactions_Studied = (Data_to_Predict.iloc[:,7].astype(str) + ' + ' + Data_to_Predict.iloc[:,8].astype(str) + "  \u2192  " + Data_to_Predict.iloc[:,9].astype(str) + ' + ' + Data_to_Predict.iloc[:,10].astype(str)) # Writing the reaction for training data introducing new column called 'reaction'
Reactions_Studied = pd.DataFrame(Reactions_Studied, columns=['Reactions'])

All_Data_with_Prediction = pd.concat([Reactions_Studied, Metals_Involved, Activation_Barrier_Comparred], axis=1)

All_Data_with_Prediction.head()


# All_Data_with_Prediction['Ea_Difference'] = All_Data_with_Prediction['Best_Model'] - All_Data_with_Prediction['Activation Barrier_eV']



All_Data_with_Prediction.to_excel('Prediction_Validation_Data_with_Descriptor.xlsx', index=False)

## Coparing the Activation Barrier Predicted on Ni and NiB

In [ ]:
NiB_Prediction = All_Data_with_Prediction[(All_Data_with_Prediction["Metal"] == 'NiB')] # Storing the predicted data of NiB surface
Ni_Prediction = All_Data_with_Prediction[(All_Data_with_Prediction["Metal"] == 'Ni')] # Storing the predicted data of Ni surface

for Model in Models:

    fig, ax = plt.subplots(nrows=1, ncols=2, sharex=True, sharey=True, figsize=(10, 4))

    fig.text(0.5, -0.7, 'Reactions', ha='center', size=18)
    fig.text(0.04, 0.5, 'Activation Barrier [eV]', va='center', rotation='vertical', size=18)

    # Subplot for Ni
    ax[0].plot(Ni_Prediction['Reactions'], Ni_Prediction['Activation Barrier_eV'], label="DFT", color = '#ffa600', linestyle = ':', marker='o')
    ax[0].plot(Ni_Prediction['Reactions'], Ni_Prediction[Model], label=f"{Model}", color = '#003f5c', linestyle = ':', marker='d')
    ax[0].set_ylim(0, 2.5)
    ax[0].set_title("Ni", size=16)
    ax[0].tick_params(axis='x', labelrotation=90, labelsize=11)
    ax[0].legend()

    # Subplot for NiB
    ax[1].plot(NiB_Prediction['Reactions'], NiB_Prediction['Activation Barrier_eV'], label="DFT", color = '#ffa600', linestyle = ':', marker='o')
    ax[1].plot(NiB_Prediction['Reactions'], NiB_Prediction[Model], label=f"{Model}", color = '#003f5c', linestyle = ':', marker='d')
    ax[1].set_ylim(0, 2.5)
    ax[1].set_title("NiB", size=16)
    ax[1].tick_params(axis='x', labelrotation=90, labelsize=11)
    ax[1].legend()

    

    plt.subplots_adjust(hspace=0.1, wspace=0.15)

    # Save with model name in filename
    filename = f"Ni NiB Comparison {Model}.jpg"
    plt.savefig(filename, dpi=1000, bbox_inches='tight')
    
    


## Comparing the Trend Observed by ML and DFT calculations

In [ ]:
NiB_Prediction = All_Data_with_Prediction[(All_Data_with_Prediction["Metal"] == 'NiB')] # Storing the predicted data of NiB surface
Ni_Prediction = All_Data_with_Prediction[(All_Data_with_Prediction["Metal"] == 'Ni')] # Storing the predicted data of Ni surface

for Model in Models:

    fig, ax = plt.subplots(nrows=1, ncols=2, sharex=True, sharey=True, figsize=(10, 4))

    fig.text(0.5, -0.7, 'Reactions', ha='center', size=18)
    fig.text(0.04, 0.5, 'Activation Barrier [eV]', va='center', rotation='vertical', size=18)

    # Subplot for Ni
    ax[0].plot(Ni_Prediction['Reactions'], Ni_Prediction['Activation Barrier_eV'], label="DFT_Ni", color = '#ffa600', linestyle = '--', marker='d')
    ax[0].plot(NiB_Prediction['Reactions'], NiB_Prediction['Activation Barrier_eV'], label="DFT_NiB", color = '#003f5c', linestyle = ':', marker='o')

    ax[0].set_ylim(0, 2.5)
    ax[0].set_title("DFT", size=16)
    ax[0].tick_params(axis='x', labelrotation=90, labelsize=11)
    ax[0].legend()

    # Subplot for NiB
    
    
    ax[1].plot(Ni_Prediction['Reactions'], Ni_Prediction[Model], label=f"{Model}_Ni", color = '#ffa600', linestyle = '--', marker='d')
    ax[1].plot(NiB_Prediction['Reactions'], NiB_Prediction[Model], label=f"{Model}_NiB", color = '#003f5c', linestyle = ':', marker='o')
    ax[1].set_ylim(0, 2.5)
    ax[1].set_title("ML", size=16)
    ax[1].tick_params(axis='x', labelrotation=90, labelsize=11)
    ax[1].legend()

    

    plt.subplots_adjust(hspace=0.1, wspace=0.15)

    # Save with model name in filename
    filename = f"DFT ML Comparison {Model}.jpg"
    plt.savefig(filename, dpi=1000, bbox_inches='tight')
    plt.show()